# 🛒 Masterclass 13: Density Clustering & Transaction Association Rules
This notebook details non-spherical clusters and basket mining:

1. **Project 1 (Scratch)**: A custom `DBSCAN` core queue scanner density clusterer from scratch.
2. **Project 2 (Applied)**: Market Basket rule extractions from transaction logs using FP-Growth.


## 📐 Part 1: Mathematical Foundations
Apriori rule metrics identify strong product associations:
$$\text{Support}(A \to B) = \frac{\text{Transactions containing } A \text{ and } B}{\text{Total Transactions}}$$
$$\text{Confidence}(A \to B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$$
$$\text{Lift}(A \to B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A) \times \text{Support}(B)}$$


In [ ]:
import numpy as np

class DBSCANScratch:
    def __init__(self, eps=0.5, min_samples=3):
        self.eps = eps
        self.min_samples = min_samples

    def _expand(self, X, labels, neighbors, cluster_id):
        queue = list(neighbors)
        i = 0
        while i < len(queue):
            pt_idx = queue[i]
            if labels[pt_idx] == -1: # Noise becomes border
                labels[pt_idx] = cluster_id
            elif labels[pt_idx] == 0 or labels[pt_idx] == -2: # Unvisited
                labels[pt_idx] = cluster_id
                dists = np.linalg.norm(X - X[pt_idx], axis=1)
                pt_neighbors = np.where(dists <= self.eps)[0]
                if len(pt_neighbors) >= self.min_samples:
                    # Add new core point neighbors to scan queue
                    for n in pt_neighbors:
                        if n not in queue: queue.append(n)
            i += 1

    def fit(self, X):
        labels = np.zeros(X.shape[0], dtype=int) - 2 # -2 = Unvisited
        cluster_id = 1
        for i in range(X.shape[0]):
            if labels[i] != -2: continue
            dists = np.linalg.norm(X - X[i], axis=1)
            neighbors = np.where(dists <= self.eps)[0]
            if len(neighbors) < self.min_samples:
                labels[i] = -1 # Noise
            else:
                labels[i] = cluster_id
                self._expand(X, labels, neighbors, cluster_id)
                cluster_id += 1
        self.labels_ = labels


## 🧪 Project 2: Association Rules via FP-Growth


In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Create transactional transaction table
df = pd.DataFrame([
    [True, True, False, False],
    [True, True, True, False],
    [True, False, False, False],
    [True, True, False, True]
], columns=['Milk', 'Bread', 'Diapers', 'Beer'])

itemsets = fpgrowth(df, min_support=0.5, use_colnames=True)
rules = association_rules(itemsets, metric='confidence', min_threshold=0.6)
print('Association rules successfully mined:')
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
